# 第13回講義 宿題

## 課題

変分オートエンコーダ（VAE）を用いて，FasionMNISTの画像を生成してみましょう


### 目標値

NLL（負の対数尤度） 235

### ルール

訓練データはx_train，テストデータはx_testで与えられます．
下のセルで指定されているx_train以外の学習データは使わないでください．

### 提出方法

- 2つのファイルを提出していただきます
    1. テストデータ (`x_test`) の再構成結果を`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第13回 深層生成モデル」を選択して**提出してください
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第13回 深層生成モデル (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）コードの内容を変更した場合は，**1と2の両方を提出し直してください**


### 評価方法

- 評価は生成画像のテストデータに対するNLL（負の対数尤度）で行います

\begin{equation}
-\sum_{i=1}^Dx_i\log\hat{x_i}+(1-x_i)\log(1-\hat{x_i})
\end{equation}

- 即時採点しLeader Boardを更新します
- 締切時の点数を最終的な評価とします

### ドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 作業ディレクトリを指定
work_dir = 'drive/MyDrive/DLBasic/HW/HW13'

### データの読み込み（このセルは修正しないでください）

In [23]:
import numpy as np
import pandas as pd
import torch

seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

# 学習データ
x_train = np.load(work_dir + '/Lecture13/data/x_train.npy')
# テストデータ
x_test = np.load(work_dir + '/Lecture13/data/x_test.npy')


class dataset(torch.utils.data.Dataset):
    def __init__(self, x_test):
        self.x_test = x_test.reshape(-1, 784).astype('float32') / 255

    def __len__(self):
        return self.x_test.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.x_test[idx], dtype=torch.float)

trainval_data = dataset(x_train)
test_data = dataset(x_test)

### VAEの実装


In [33]:
batch_size = 64

val_size = 10000
train_size = len(trainval_data) - val_size

train_data, val_data = torch.utils.data.random_split(trainval_data, [train_size, val_size])

dataloader_train = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

dataloader_valid = torch.utils.data.DataLoader(
    val_data,
    batch_size=batch_size,
    shuffle=False
)

dataloader_test = torch.utils.data.DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [38]:
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torch.nn.functional as F
from typing import Tuple

device = 'cuda'


# torch.log(0)によるnanを防ぐ
def torch_log(x):
    return torch.log(torch.clamp(x, min=1e-10))

class VAE(nn.Module):
    def __init__(self, z_dim: int) -> None:
        super().__init__()

        self.dense_enc1 = nn.Linear(28*28, 600)
        self.dense_enc2 = nn.Linear(600, 400)
        self.dense_encmean = nn.Linear(400, z_dim)
        self.dense_encvar = nn.Linear(400, z_dim)

        self.dense_dec1 = nn.Linear(z_dim, 400)
        self.dense_dec2 = nn.Linear(400, 600)
        self.dense_dec3 = nn.Linear(600, 28*28)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def _encoder(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = F.relu(self.dense_enc1(x))
        x = F.relu(self.dense_enc2(x))
        mean = self.dense_encmean(x)
        std = F.softplus(self.dense_encvar(x)) + 1e-8

        return mean, std

    def _sample_z(self, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
        if self.training:
            epsilon = torch.randn(mean.shape).to(device)
            return mean + std * epsilon
        else:
            return mean

    def _decoder(self, z: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.dense_dec1(z))
        x = F.relu(self.dense_dec2(x))
        x = torch.sigmoid(self.dense_dec3(x))

        return x

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mean, std = self._encoder(x)
        z = self._sample_z(mean, std)
        x = self._decoder(z)
        return x, (mean, std)

    def loss(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mean, std = self._encoder(x)

        KL = -0.5 * torch.mean(torch.sum(1 + torch_log(std**2) - mean**2 - std**2, dim=1))

        z = self._sample_z(mean, std)
        y = self._decoder(z)

        reconstruction = torch.mean(torch.sum(x * torch_log(y) + (1 - x) * torch_log(1 - y), dim=1))

        return KL, -reconstruction


## past

In [39]:

# 232.141
# class VAE(nn.Module):
#     def __init__(self, z_dim: int) -> None:
#         super().__init__()

#         self.dense_enc1 = nn.Linear(28*28, 800)
#         self.dense_enc2 = nn.Linear(800, 600)
#         self.dense_enc3 = nn.Linear(600, 400)
#         self.dense_encmean = nn.Linear(400, z_dim)
#         self.dense_encvar = nn.Linear(400, z_dim)

#         self.dense_dec1 = nn.Linear(z_dim, 400)
#         self.dense_dec2 = nn.Linear(400, 600)
#         self.dense_dec3 = nn.Linear(600, 800)
#         self.dense_dec4 = nn.Linear(800, 28*28)

#         # Dropout
#         self.dropout = nn.Dropout(0.1)

#         self._initialize_weights()

#     def _initialize_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.xavier_uniform_(m.weight)
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)

#     def _encoder(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         x = F.relu(self.dense_enc1(x))
#         x = self.dropout(x)
#         x = F.relu(self.dense_enc2(x))
#         x = self.dropout(x)
#         x = F.relu(self.dense_enc3(x))
#         mean = self.dense_encmean(x)
#         std = F.softplus(self.dense_encvar(x)) + 1e-8  # 数値安定性のために小さな値を追加

#         return mean, std

#     def _sample_z(self, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
#         if self.training:
#             epsilon = torch.randn(mean.shape).to(device)
#             return mean + std * epsilon
#         else:
#             return mean

#     def _decoder(self, z: torch.Tensor) -> torch.Tensor:
#         x = F.relu(self.dense_dec1(z))
#         x = self.dropout(x)
#         x = F.relu(self.dense_dec2(x))
#         x = self.dropout(x)
#         x = F.relu(self.dense_dec3(x))
#         x = torch.sigmoid(self.dense_dec4(x))

#         return x

#     def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         mean, std = self._encoder(x)
#         z = self._sample_z(mean, std)
#         x = self._decoder(z)
#         return x, (mean, std)

#     def loss(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         mean, std = self._encoder(x)

#         KL = -0.5 * torch.mean(torch.sum(1 + torch_log(std**2) - mean**2 - std**2, dim=1))

#         z = self._sample_z(mean, std)
#         y = self._decoder(z)

#         reconstruction = torch.mean(torch.sum(x * torch_log(y) + (1 - x) * torch_log(1 - y), dim=1))

#         return KL, -reconstruction

# 230.814
# class VAE(nn.Module):
#     def __init__(self, z_dim: int) -> None:
#         super().__init__()

#         # Encoder, xを入力にガウス分布のパラメータmu, sigmaを出力
#         self.dense_enc1 = nn.Linear(28*28, 500)  # 隠れ層を500に増加
#         self.dense_enc2 = nn.Linear(500, 500)    # 2層目も500に
#         self.dense_enc3 = nn.Linear(500, 400)    # 3層目を追加
#         self.dense_encmean = nn.Linear(400, z_dim)
#         self.dense_encvar = nn.Linear(400, z_dim)

#         # Decoder, zを入力にベルヌーイ分布のパラメータlambdaを出力
#         self.dense_dec1 = nn.Linear(z_dim, 400)
#         self.dense_dec2 = nn.Linear(400, 500)
#         self.dense_dec3 = nn.Linear(500, 500)
#         self.dense_dec4 = nn.Linear(500, 28*28)

#         # 重み初期化
#         self._initialize_weights()

#     def _initialize_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)

#     def _encoder(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         x = F.relu(self.dense_enc1(x))
#         x = F.relu(self.dense_enc2(x))
#         x = F.relu(self.dense_enc3(x))
#         mean = self.dense_encmean(x)
#         std = F.softplus(self.dense_encvar(x))

#         return mean, std

#     def _sample_z(self, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
#         if self.training:
#             # 再パラメータ化トリック．この乱数は計算グラフで勾配の通り道に無い．
#             epsilon = torch.randn(mean.shape).to(device)
#             return mean + std * epsilon
#         else:
#             return mean

#     def _decoder(self, z: torch.Tensor) -> torch.Tensor:
#         x = F.relu(self.dense_dec1(z))
#         x = F.relu(self.dense_dec2(x))
#         x = F.relu(self.dense_dec3(x))
#         # 出力が0~1になるようにsigmoid
#         x = torch.sigmoid(self.dense_dec4(x))

#         return x

#     def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         mean, std = self._encoder(x)
#         z = self._sample_z(mean, std)
#         x = self._decoder(z)
#         return x, (mean, std)

#     def loss(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
#         mean, std = self._encoder(x)

#         # KL loss(正則化項)の計算. mean, stdは (batch_size , z_dim)
#         KL = -0.5 * torch.mean(torch.sum(1 + torch_log(std**2) - mean**2 - std**2, dim=1))

#         z = self._sample_z(mean, std)
#         y = self._decoder(z)

#         # reconstruction loss(負の再構成誤差)の計算. x, yともに (batch_size , 784)
#         reconstruction = torch.mean(torch.sum(x * torch_log(y) + (1 - x) * torch_log(1 - y), dim=1))

#         return KL, -reconstruction

## train

In [40]:
z_dim = 50
n_epochs = 100
lr = 3e-4

model = VAE(z_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)
best_val_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(n_epochs):
    model.train()
    losses = []
    KL_losses = []
    reconstruction_losses = []

    for x in dataloader_train:
        x = x.to(device)
        optimizer.zero_grad()

        KL_loss, reconstruction_loss = model.loss(x)
        loss = KL_loss + reconstruction_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        losses.append(loss.item() / len(x))
        KL_losses.append(KL_loss.item() / len(x))
        reconstruction_losses.append(reconstruction_loss.item() / len(x))

    losses_val = []
    model.eval()
    with torch.no_grad():
        for x in dataloader_valid:
            x = x.to(device)
            KL_loss, reconstruction_loss = model.loss(x)
            loss = KL_loss + reconstruction_loss
            losses_val.append(loss.cpu().detach().numpy())

    # scheduler.step()
    current_val_loss = np.average(losses_val)
    scheduler.step(current_val_loss)

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_vae_model.pth')
    else:
        patience_counter += 1

    if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            torch.save(model.state_dict(), 'best_vae_model.pth')

    print('EPOCH:%d, Train Lower Bound:%lf, (%lf, %lf), Valid Lower Bound:%lf' %
              (epoch+1, np.average(losses), np.average(KL_losses), np.average(reconstruction_losses), current_val_loss))

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

EPOCH:1, Train Lower Bound:4.558184, (0.213394, 4.344789), Valid Lower Bound:268.459564
EPOCH:2, Train Lower Bound:4.050918, (0.211288, 3.839630), Valid Lower Bound:253.996811
EPOCH:3, Train Lower Bound:3.934233, (0.201008, 3.733225), Valid Lower Bound:247.444763
EPOCH:4, Train Lower Bound:3.880029, (0.200201, 3.679828), Valid Lower Bound:244.444443
EPOCH:5, Train Lower Bound:3.847990, (0.202383, 3.645607), Valid Lower Bound:241.424667
EPOCH:6, Train Lower Bound:3.824896, (0.204837, 3.620059), Valid Lower Bound:241.376511
EPOCH:7, Train Lower Bound:3.807563, (0.207242, 3.600321), Valid Lower Bound:239.111618
EPOCH:8, Train Lower Bound:3.793930, (0.209154, 3.584776), Valid Lower Bound:238.479752
EPOCH:9, Train Lower Bound:3.782027, (0.210308, 3.571719), Valid Lower Bound:237.143463
EPOCH:10, Train Lower Bound:3.772378, (0.211589, 3.560789), Valid Lower Bound:236.173660
EPOCH:11, Train Lower Bound:3.764615, (0.212504, 3.552111), Valid Lower Bound:236.298523
EPOCH:12, Train Lower Bound:3.

In [41]:
import csv

model.load_state_dict(torch.load('best_vae_model.pth'))

sample_x = []
answer = []
model.eval()
for x in dataloader_test:

    x = x.to(device)

    y, _ = model(x)

    y = y.tolist()

    sample_x.extend(y)

with open(work_dir + '/Lecture13/submission_pred.csv', 'w') as file:
    writer = csv.writer(file, lineterminator='\n')
    writer.writerows(sample_x)
file.close()